In [1]:
import io
import re
import requests
import pandas as pd

URL = "https://www.iposcoop.com/ipos-recently-filed/"
HEADERS = {"User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                          "AppleWebKit/537.36 (KHTML, like Gecko) "
                          "Chrome/120.0 Safari/537.36")}

resp = requests.get(URL, headers=HEADERS, timeout=30)
resp.raise_for_status()
print("status:", resp.status_code, "len:", len(resp.text))

tables = pd.read_html(io.StringIO(resp.text))
print("tables found:", len(tables))
for i, t in enumerate(tables):
    print(i, t.shape, list(t.columns)[:6])

status: 200 len: 332912
tables found: 1
0 (500, 10) ['File Date', 'Company', 'Symbol', 'Managers', 'Shares (millions)', 'Price Low']


In [2]:
raw = tables[0].copy()
print(list(raw.columns))
raw.head()

['File Date', 'Company', 'Symbol', 'Managers', 'Shares (millions)', 'Price Low', 'Price High', 'Est $ Vol (millions)', 'Expected To Trade', 'SCOOP Rating']


,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,2026-10-15,City Therapeutics,CTY,Goldman Sachs/Jefferies/Stifel/Oppenheimer & Co.,11.11,$18.00,$18.00,$199.98,Priced,S/O
1,2026-10-15,TCGX Acquisition Corp. II (Stock-Only),TCXB,Guggenheim Securities,10.00,$10.00,$10.00,$100.00,Priced,S/O
2,2026-10-14,Amaero (Revived),AMRO,Baird/Lake Street,4.71,$4.25,$4.25,$20.02,Priced,S/O
3,2026-10-14,"BioVentrix, Inc. (Withdrawn)",BVXX,The Benchmark Company,0.00,NaN,NaN,$17.00,Withdrawn,S/O
4,2026-10-14,Bitari,BIAI,US Tiger Securities,4.29,$7.00,$7.00,$30.03,Week of,S/O


In [3]:
trade_col = [c for c in raw.columns if "expected" in str(c).lower()][0]
print("status column:", trade_col)
print(raw[trade_col].value_counts())

wd = raw[raw[trade_col].astype(str).str.strip().str.lower() == "withdrawn"].copy()
print("\nwithdrawn rows:", len(wd))
wd.head()

status column: Expected To Trade
Expected To Trade
Priced       255
TBA          188
Withdrawn     36
Week of        8
Wednesday      4
Thursday       3
Postponed      3
Tuesday        2
Friday         1
Name: count, dtype: int64

withdrawn rows: 36


,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
3,2026-10-14,"BioVentrix, Inc. (Withdrawn)",BVXX,The Benchmark Company,0.00,NaN,NaN,$17.00,Withdrawn,S/O
29,2026-10-07,Siyata PTT (Withdrawn – Direct Listing),PTT,Digital Offering LLC (Financial Advisor),6.11,$7.00,$7.00,$42.77,Withdrawn,S/O
57,2026-09-25,Holtec Nuclear (Withdrawn),HNUC,J.P.Morgan/Guggenheim Securities/Goldman Sachs...,50.00,$15.00,$18.00,$825.00,Withdrawn,S/O
72,2026-09-18,Rise Smart Group (Withdrawn),RSHL,American Trust Investment Services/Prime Numbe...,1.80,$4.00,$4.00,$7.20,Withdrawn,S/O
74,2026-09-17,C2 Capital Group (Withdrawn),CCLV,Dominari Securities,3.75,$4.00,$5.00,$16.88,Withdrawn,S/O


In [4]:
def classify(name: str) -> str:
    n = str(name)
    if "Technologies" in n:
        return "Technologies"
    if ("Acquisition Corp" in n or "Acquisition Corporation" in n or "Corp" in n):
        return "Acquisition Corp"
    if "Inc" in n or "Incorporated" in n:
        return "Inc."
    if "Group" in n:
        return "Group"
    if "Ltd" in n or "Limited" in n:
        return "Limited"
    if "Holdings" in n or "Holding" in n:
        return "Holdings"
    return "Other"

name_col = [c for c in wd.columns if "company" in str(c).lower()][0]
wd["Company Type"] = wd[name_col].apply(classify)
wd[[name_col, "Company Type"]].to_string()

'                                             Company      Company Type\n3                       BioVentrix, Inc. (Withdrawn)              Inc.\n29           Siyata PTT (Withdrawn – Direct Listing)             Other\n57                        Holtec Nuclear (Withdrawn)             Other\n72                      Rise Smart Group (Withdrawn)             Group\n74                      C2 Capital Group (Withdrawn)             Group\n82                            OTSAW Ltd. (Withdrawn)           Limited\n86                   Motive Technologies (Withdrawn)      Technologies\n89                     Idea Tech Holding (Withdrawn)          Holdings\n95            Hornbeck Offshore Services (Withdrawn)             Other\n97             Coolbit Technologies Ltd. (Withdrawn)      Technologies\n104        Timber Road Acquisition Corp. (Withdrawn)  Acquisition Corp\n113                     pan-Africa Corp. (Withdrawn)  Acquisition Corp\n125                    Living Homeopathy (Withdrawn)           

In [5]:
wd.head()

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating,Company Type
3,2026-10-14,"BioVentrix, Inc. (Withdrawn)",BVXX,The Benchmark Company,0.00,NaN,NaN,$17.00,Withdrawn,S/O,Inc.
29,2026-10-07,Siyata PTT (Withdrawn – Direct Listing),PTT,Digital Offering LLC (Financial Advisor),6.11,$7.00,$7.00,$42.77,Withdrawn,S/O,Other
57,2026-09-25,Holtec Nuclear (Withdrawn),HNUC,J.P.Morgan/Guggenheim Securities/Goldman Sachs...,50.00,$15.00,$18.00,$825.00,Withdrawn,S/O,Other
72,2026-09-18,Rise Smart Group (Withdrawn),RSHL,American Trust Investment Services/Prime Numbe...,1.80,$4.00,$4.00,$7.20,Withdrawn,S/O,Group
74,2026-09-17,C2 Capital Group (Withdrawn),CCLV,Dominari Securities,3.75,$4.00,$5.00,$16.88,Withdrawn,S/O,Group


In [6]:
def to_num(x):
    """'$8.00' -> 8.0 ; '-', '', NaN -> None"""
    if pd.isna(x):
        return None
    s = str(x).replace("$", "").replace(",", "").strip()
    if s in ("", "-", "–", "—", "N/A", "n/a"):
        return None
    m = re.search(r"-?\d+(?:\.\d+)?", s)
    return float(m.group()) if m else None

low_col  = [c for c in wd.columns if "low"  in str(c).lower()][0]
high_col = [c for c in wd.columns if "high" in str(c).lower()][0]

wd["price_low_num"]  = wd[low_col].apply(to_num)
wd["price_high_num"] = wd[high_col].apply(to_num)
wd["Avg_price"] = wd[["price_low_num", "price_high_num"]].mean(axis=1)

print("missing Avg_price:", wd["Avg_price"].isna().sum())
wd[[low_col, high_col, "Avg_price"]].head(10)

missing Avg_price: 7


,Price Low,Price High,Avg_price
3,NaN,NaN,NaN
29,$7.00,$7.00,7.0
57,$15.00,$18.00,16.5
72,$4.00,$4.00,4.0
74,$4.00,$5.00,4.5
82,$4.50,$5.50,5.0
86,NaN,NaN,NaN
89,$4.00,$5.00,4.5
95,NaN,NaN,NaN
97,$4.00,$5.00,4.5


In [7]:
shares_col = [c for c in wd.columns if "shares" in str(c).lower()][0]
vol_col    = [c for c in wd.columns if "vol"    in str(c).lower()][0]
print("shares:", shares_col, "| vol:", vol_col)

wd["Shares_num"] = wd[shares_col].apply(to_num)
wd["Est_vol_num"] = wd[vol_col].apply(to_num)

print("missing shares:", wd["Shares_num"].isna().sum(),
      "| missing vol:", wd["Est_vol_num"].isna().sum())
wd[[shares_col, "Shares_num", vol_col, "Est_vol_num"]].head(10)

shares: Shares (millions) | vol: Est $ Vol (millions)
missing shares: 0 | missing vol: 0


,Shares (millions),Shares_num,Est $ Vol (millions),Est_vol_num
3,0.00,0.00,$17.00,17.00
29,6.11,6.11,$42.77,42.77
57,50.00,50.00,$825.00,825.00
72,1.80,1.80,$7.20,7.20
74,3.75,3.75,$16.88,16.88
82,4.40,4.40,$22.00,22.00
86,0.00,0.00,$100.00,100.00
89,2.00,2.00,$9.00,9.00
95,0.00,0.00,$100.00,100.00
97,5.00,5.00,$22.50,22.50


In [8]:
zero_shares = wd[wd["Shares_num"] == 0]
print("rows with 0 shares:", len(zero_shares))
zero_shares[[name_col, "Shares_num", "Avg_price", "Est_vol_num"]]

rows with 0 shares: 7


,Company,Shares_num,Avg_price,Est_vol_num
3,"BioVentrix, Inc. (Withdrawn)",0.0,NaN,17.0
86,Motive Technologies (Withdrawn),0.0,NaN,100.0
95,Hornbeck Offshore Services (Withdrawn),0.0,NaN,100.0
222,Kilcoy Global Foods Ltd. (Withdrawn),0.0,NaN,100.0
310,Xinxu Copper Industry Technology Ltd.,0.0,NaN,15.0
370,Going International Holding Company Limited,0.0,NaN,20.0
407,Invea Therapeutics (Withdrawn),0.0,NaN,75.0


In [9]:
computed = wd["Shares_num"] * wd["Avg_price"]
wd["Shares_offered_value"] = computed.fillna(wd["Est_vol_num"])

print("rows using computed value:", computed.notna().sum())
print("rows falling back to Est $ Vol:", computed.isna().sum())
print("still null after fallback:", wd["Shares_offered_value"].isna().sum())
wd[[name_col, "Company Type", "Shares_num", "Avg_price",
    "Est_vol_num", "Shares_offered_value"]].to_string()

rows using computed value: 29
rows falling back to Est $ Vol: 7
still null after fallback: 0


'                                             Company      Company Type  Shares_num  Avg_price  Est_vol_num  Shares_offered_value\n3                       BioVentrix, Inc. (Withdrawn)              Inc.        0.00        NaN        17.00               17.0000\n29           Siyata PTT (Withdrawn – Direct Listing)             Other        6.11       7.00        42.77               42.7700\n57                        Holtec Nuclear (Withdrawn)             Other       50.00      16.50       825.00              825.0000\n72                      Rise Smart Group (Withdrawn)             Group        1.80       4.00         7.20                7.2000\n74                      C2 Capital Group (Withdrawn)             Group        3.75       4.50        16.88               16.8750\n82                            OTSAW Ltd. (Withdrawn)           Limited        4.40       5.00        22.00               22.0000\n86                   Motive Technologies (Withdrawn)      Technologies        0.00       

In [10]:
agg = (wd.groupby("Company Type")["Shares_offered_value"]
         .sum()
         .sort_values(ascending=False))
print(agg.round(2).to_string())

top = agg.idxmax()
print(f"\nHighest: {top} — ${agg.max():,.2f} million")

Company Type
Other               1158.21
Acquisition Corp     499.98
Inc.                 368.00
Holdings             311.66
Limited              213.25
Technologies         184.90
Group                 56.58

Highest: Other — $1,158.21 million


In [11]:
wd["File Date"] = pd.to_datetime(wd["File Date"], errors="coerce")
sub = wd[wd["File Date"] < "2026-09-11"]
print("rows:", len(sub))
print(sub.groupby("Company Type")["Shares_offered_value"].sum()
         .sort_values(ascending=False).round(2).to_string())

rows: 30
Company Type
Acquisition Corp    499.98
Inc.                351.00
Holdings            311.66
Other               290.44
Limited             191.25
Technologies        184.90
Group                32.50
